In [1]:
!pip install -q gradio pandas numpy scikit-learn matplotlib seaborn
print("✅ Architecture Dependencies Installed Successfully!")

✅ Architecture Dependencies Installed Successfully!


In [2]:
# --- CELL 2: KNOWLEDGE BASE & FEATURE EXTRACTION ---
import pandas as pd
import numpy as np

print("⚙️ Initializing Digital Matchmaker Database...")

# Creating a dataset of items with specific feature profiles
data = {
    'Item_Name': ['Quantum Desktop PC', 'Nomad Ultrabook', 'GamerX Console', 'Creator Pro Studio', 'Cloud Server Node', 'VR Headset'],
    'Performance': [0.95, 0.50, 0.85, 0.90, 0.99, 0.70],
    'Portability': [0.10, 0.95, 0.40, 0.30, 0.05, 0.80],
    'Gaming':      [0.90, 0.20, 0.99, 0.60, 0.10, 0.95],
    'Productivity':[0.85, 0.90, 0.30, 0.99, 0.90, 0.40]
}

# Load into a Pandas DataFrame
df_items = pd.DataFrame(data)

# Extract just the numerical features to create our Vector Space
feature_matrix = df_items[['Performance', 'Portability', 'Gaming', 'Productivity']].values

print("📊 Feature Vectors Mapped. Database Ready.")
df_items

⚙️ Initializing Digital Matchmaker Database...
📊 Feature Vectors Mapped. Database Ready.


,Item_Name,Performance,Portability,Gaming,Productivity
0,Quantum Desktop PC,0.95,0.10,0.90,0.85
1,Nomad Ultrabook,0.50,0.95,0.20,0.90
2,GamerX Console,0.85,0.40,0.99,0.30
3,Creator Pro Studio,0.90,0.30,0.60,0.99
4,Cloud Server Node,0.99,0.05,0.10,0.90
5,VR Headset,0.70,0.80,0.95,0.40


In [3]:
# --- CELL 3: COSINE SIMILARITY ALGORITHM ---
from sklearn.metrics.pairwise import cosine_similarity

def calculate_similarity(user_perf, user_port, user_game, user_prod):
    """
    Takes user input parameters, converts them into a 1D vector,
    and calculates the Cosine Similarity against all items in the database.
    """
    # 1. Map user input to a feature vector
    user_vector = np.array([[user_perf, user_port, user_game, user_prod]])

    # 2. Calculate Cosine Similarity
    similarity_scores = cosine_similarity(user_vector, feature_matrix)[0]

    # 3. Append scores to the dataframe
    df_results = df_items.copy()
    df_results['Match_Score'] = similarity_scores

    # 4. Sort and return the top 3 recommendations
    top_matches = df_results.sort_values(by='Match_Score', ascending=False).head(3)
    return top_matches

print("🧠 Similarity Engine Initialized.")

🧠 Similarity Engine Initialized.


In [ ]:
# --- CELL 4: DYNAMIC FRONTEND & VISUALIZATIONS ---
import gradio as gr
import matplotlib.pyplot as plt
import seaborn as sns

def matchmaker_pipeline(perf, port, game, prod):
    # Retrieve top matches from the engine
    top_matches = calculate_similarity(perf, port, game, prod)
    best_match = top_matches.iloc[0]

    # --- VISUALIZATION GENERATION ---
    # Create a 1x2 grid for our professional dashboard
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Plot 1: Top 3 Similarity Scores
    sns.barplot(data=top_matches, x='Match_Score', y='Item_Name', ax=axes[0], palette='mako')
    axes[0].set_xlim(0, 1.0)
    axes[0].set_title("Top 3 Recommendations (Cosine Similarity)")
    axes[0].set_xlabel("Match Alignment Score")
    axes[0].set_ylabel("")

    # Plot 2: Vector Alignment (User Profile vs. Best Match Profile)
    categories = ['Performance', 'Portability', 'Gaming', 'Productivity']
    user_profile = [perf, port, game, prod]
    best_match_profile = [best_match['Performance'], best_match['Portability'], best_match['Gaming'], best_match['Productivity']]

    x = np.arange(len(categories))
    width = 0.35

    # Grouped bar chart to visually prove vector mapping
    axes[1].bar(x - width/2, user_profile, width, label='Your Target Profile', color='#ff7f0e')
    axes[1].bar(x + width/2, best_match_profile, width, label=f"Best Match ({best_match['Item_Name']})", color='#1f77b4')

    axes[1].set_title("Profile Alignment: Target vs Actual Match")
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(categories)
    axes[1].set_ylim(0, 1.0)
    axes[1].legend()

    plt.tight_layout()

    # --- TEXT OUTPUT GENERATION ---
    text_output = "🏆 **YOUR DIGITAL MATCHES:**\n\n"
    for index, row in top_matches.iterrows():
        text_output += f"• **{row['Item_Name']}** (Alignment: {row['Match_Score']*100:.1f}%)\n"

    return text_output, fig

# --- GRADIO UI CONSTRUCTION ---
with gr.Blocks(theme=gr.themes.Soft(primary_hue="indigo")) as ui:
    gr.Markdown("<h1 style='text-align: center;'>🧠 DecodeLabs | Digital Matchmaker Engine</h1>")
    gr.Markdown("<p style='text-align: center;'>Adjust your desired features. The AI will extract your profile vector, calculate cosine similarity against our database, and output flawless, relevant matches.</p>")

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 🎛️ Input Preferences")
            perf_slider = gr.Slider(0.0, 1.0, value=0.5, step=0.05, label="Raw Power & Performance")
            port_slider = gr.Slider(0.0, 1.0, value=0.5, step=0.05, label="Portability & Battery")
            game_slider = gr.Slider(0.0, 1.0, value=0.5, step=0.05, label="Gaming & Graphics")
            prod_slider = gr.Slider(0.0, 1.0, value=0.5, step=0.05, label="Work & Productivity")

            trigger_btn = gr.Button("Calculate Similarity & Map Vectors 🚀", variant="primary")

        with gr.Column(scale=2):
            gr.Markdown("### 📊 AI Output Logic")
            text_result = gr.Markdown()
            plot_result = gr.Plot(label="Real-Time Vector Analytics")

    # Bind the backend function to the UI components
    trigger_btn.click(
        fn=matchmaker_pipeline,
        inputs=[perf_slider, port_slider, game_slider, prod_slider],
        outputs=[text_result, plot_result]
    )

print("\n🌐 Launching Responsive Web Interface...")
# Launch with share=True to generate a public deployment link
ui.launch(share=True, debug=True)

/tmp/ipykernel_519/3165542744.py:50: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(primary_hue="indigo")) as ui:



🌐 Launching Responsive Web Interface...
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://6b6a3cf9eda782f7dc.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/tmp/ipykernel_519/3165542744.py:16: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=top_matches, x='Match_Score', y='Item_Name', ax=axes[0], palette='mako')
